# Semana 5: Modelos Clásicos para NLP: Naive Bayes y SVM
## Notebook Conceptual (NB1) – Clasificación con Datos Dummy

**Propósito:** Aplicar algoritmos clásicos de machine learning a tareas de clasificación de textos y comparar su rendimiento con enfoques modernos.

**Docente:** Carlos César Sánchez Coronel

**Objetivos de aprendizaje:**
- Calcular probabilidades condicionales para Naive Bayes manualmente.
- Implementar un clasificador Naive Bayes desde cero usando frecuencias.
- Entrenar GaussianNB y MultinomialNB con sklearn y comparar.
- Visualizar la frontera de decisión de SVM lineal en un espacio reducido.

---

## 0. Configuración Inicial

Importamos las librerías necesarias.

In [ ]:
# Importamos librerías
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter, defaultdict

# Scikit-learn
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.naive_bayes import GaussianNB, MultinomialNB, BernoulliNB
from sklearn.svm import SVC, LinearSVC
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Configuración de visualización
%matplotlib inline
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

# Semilla
np.random.seed(42)

print("Librerías importadas correctamente.")

---
## 1. Corpus Dummy para Clasificación

Creamos un conjunto pequeño de textos inventados con dos categorías: **deportes** y **política**.

In [ ]:
# Definimos el corpus
corpus = [
    "el equipo ganó el partido de fútbol",
    "el jugador anotó un gol espectacular",
    "la pelota está en la cancha",
    "el estadio estaba lleno de aficionados",
    "el presidente firmó un nuevo decreto",
    "los diputados debatieron la ley",
    "las elecciones serán el próximo mes",
    "el gobierno anunció nuevas medidas"
]

# Etiquetas: 0 = deportes, 1 = política
y = [0, 0, 0, 0, 1, 1, 1, 1]

print("=== CORPUS ===")
for i, (doc, label) in enumerate(zip(corpus, y)):
    clase = "deportes" if label == 0 else "política"
    print(f"{i+1}. [{clase}] {doc}")

---
## 2. Naive Bayes Manual

### 2.1. Cálculo de probabilidades condicionales

Implementamos Naive Bayes desde cero usando frecuencias.

In [ ]:
class NaiveBayesManual:
    """Clasificador Naive Bayes manual para texto."""
    
    def __init__(self, alpha=1.0):
        self.alpha = alpha  # suavizado Laplace
        self.class_probs = {}
        self.word_probs = {}
        self.vocab = set()
        self.classes = None
    
    def fit(self, X, y):
        """
        X: lista de documentos (strings)
        y: lista de etiquetas
        """
        self.classes = np.unique(y)
        n_docs = len(y)
        
        # Tokenizar documentos
        tokenized_docs = [doc.split() for doc in X]
        
        # Crear vocabulario
        for doc in tokenized_docs:
            self.vocab.update(doc)
        vocab_size = len(self.vocab)
        
        # Calcular probabilidades a priori de clases
        for c in self.classes:
            self.class_probs[c] = np.sum(y == c) / n_docs
        
        # Contar frecuencias de palabras por clase
        word_counts = {c: defaultdict(int) for c in self.classes}
        for doc, label in zip(tokenized_docs, y):
            for word in doc:
                word_counts[label][word] += 1
        
        # Calcular probabilidades condicionales con suavizado Laplace
        for c in self.classes:
            total_words_c = sum(word_counts[c].values())
            self.word_probs[c] = {}
            for word in self.vocab:
                count = word_counts[c][word]
                self.word_probs[c][word] = (count + self.alpha) / (total_words_c + self.alpha * vocab_size)
    
    def predict_proba(self, X):
        """Devuelve probabilidades logarítmicas para cada clase."""
        tokenized_docs = [doc.split() for doc in X]
        predictions = []
        
        for doc in tokenized_docs:
            log_probs = {}
            for c in self.classes:
                # Log probabilidad a priori
                log_prob = np.log(self.class_probs[c])
                # Sumar log probabilidades de palabras
                for word in doc:
                    if word in self.word_probs[c]:
                        log_prob += np.log(self.word_probs[c][word])
                    # Si la palabra no está en vocabulario (no debería ocurrir porque usamos el mismo vocabulario)
                    # pero por si acaso, usamos una probabilidad muy pequeña
                    else:
                        log_prob += np.log(1e-10)
                log_probs[c] = log_prob
            predictions.append(log_probs)
        return predictions
    
    def predict(self, X):
        probas = self.predict_proba(X)
        return [max(p.items(), key=lambda x: x[1])[0] for p in probas]

# Entrenamos el modelo manual
nb_manual = NaiveBayesManual(alpha=1.0)
nb_manual.fit(corpus, y)

# Mostramos las probabilidades aprendidas
print("Probabilidades a priori:")
for c, prob in nb_manual.class_probs.items():
    clase = "deportes" if c == 0 else "política"
    print(f"  P({clase}) = {prob:.3f}")

print("\nProbabilidades de palabras (primeras 5 palabras de cada clase):")
for c in nb_manual.classes:
    clase = "deportes" if c == 0 else "política"
    print(f"\n{clase}:")
    # Ordenar palabras por probabilidad
    top_words = sorted(nb_manual.word_probs[c].items(), key=lambda x: x[1], reverse=True)[:5]
    for word, prob in top_words:
        print(f"  {word}: {prob:.4f}")

### 2.2. Prueba del clasificador manual

In [ ]:
# Textos de prueba
test_texts = [
    "el jugador anotó un gol en el estadio",
    "el presidente firmó una ley",
    "la pelota y el partido"
]

print("=== Predicciones con Naive Bayes Manual ===")
for text in test_texts:
    pred = nb_manual.predict([text])[0]
    clase = "deportes" if pred == 0 else "política"
    probas = nb_manual.predict_proba([text])[0]
    print(f"\nTexto: '{text}'")
    print(f"  Predicción: {clase}")
    for c, logprob in probas.items():
        clase_n = "deportes" if c == 0 else "política"
        print(f"    log P({clase_n}) = {logprob:.4f}")

---
## 3. Naive Bayes con scikit-learn

### 3.1. MultinomialNB (el más común para texto)

In [ ]:
# Vectorizamos el corpus
vectorizer = CountVectorizer()
X_vec = vectorizer.fit_transform(corpus)

# Entrenamos MultinomialNB
mnb = MultinomialNB(alpha=1.0)
mnb.fit(X_vec, y)

# Probamos con los mismos textos de prueba
X_test_vec = vectorizer.transform(test_texts)
pred_mnb = mnb.predict(X_test_vec)
proba_mnb = mnb.predict_proba(X_test_vec)

print("=== MultinomialNB ===")
for i, text in enumerate(test_texts):
    clase = "deportes" if pred_mnb[i] == 0 else "política"
    print(f"\nTexto: '{text}'")
    print(f"  Predicción: {clase}")
    print(f"  Probabilidades: deportes={proba_mnb[i][0]:.4f}, política={proba_mnb[i][1]:.4f}")

### 3.2. GaussianNB (para características continuas, no ideal para texto crudo)

In [ ]:
# GaussianNB espera características numéricas continuas
# Usaremos TF-IDF y luego convertiremos a denso (no recomendado para texto real pero para demostración)
tfidf_vectorizer = TfidfVectorizer()
X_tfidf = tfidf_vectorizer.fit_transform(corpus).toarray()

gnb = GaussianNB()
gnb.fit(X_tfidf, y)

X_test_tfidf = tfidf_vectorizer.transform(test_texts).toarray()
pred_gnb = gnb.predict(X_test_tfidf)

print("=== GaussianNB ===")
for i, text in enumerate(test_texts):
    clase = "deportes" if pred_gnb[i] == 0 else "política"
    print(f"\nTexto: '{text}'")
    print(f"  Predicción: {clase}")

### 3.3. Comparación de resultados

In [ ]:
comparacion = pd.DataFrame({
    'Texto': test_texts,
    'Manual': [nb_manual.predict([text])[0] for text in test_texts],
    'MultinomialNB': pred_mnb,
    'GaussianNB': pred_gnb
})

# Mapear números a nombres de clase
map_clase = {0: 'deportes', 1: 'política'}
for col in ['Manual', 'MultinomialNB', 'GaussianNB']:
    comparacion[col] = comparacion[col].map(map_clase)

print("=== Comparación de clasificadores Naive Bayes ===")
comparacion

---
## 4. SVM y Visualización de Fronteras de Decisión

Para visualizar la frontera de SVM, necesitamos reducir la dimensionalidad a 2D usando PCA sobre las representaciones TF-IDF.

In [ ]:
# Vectorizamos con TF-IDF
tfidf = TfidfVectorizer(max_features=50)  # limitamos para visualización
X_tfidf_vis = tfidf.fit_transform(corpus)

# Reducimos a 2D con PCA
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_tfidf_vis.toarray())

# Entrenamos SVM lineal
svm = LinearSVC(random_state=42, C=1.0)
svm.fit(X_pca, y)

# Función para visualizar la frontera
def plot_decision_boundary(X, y, model, title):
    # Crear malla
    x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
    y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 200),
                         np.linspace(y_min, y_max, 200))
    
    # Predecir sobre la malla
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)
    
    # Dibujar contorno
    plt.contourf(xx, yy, Z, alpha=0.3, cmap='bwr')
    
    # Dibujar puntos
    colors = ['blue' if label == 0 else 'red' for label in y]
    plt.scatter(X[:, 0], X[:, 1], c=colors, edgecolors='k', s=100)
    
    # Añadir etiquetas de documentos (índices)
    for i, (x, y_) in enumerate(X):
        plt.annotate(str(i), (x, y_), fontsize=9, ha='center')
    
    plt.xlabel('Componente Principal 1')
    plt.ylabel('Componente Principal 2')
    plt.title(title)
    plt.grid(True)

plt.figure(figsize=(10, 8))
plot_decision_boundary(X_pca, y, svm, "SVM Lineal en espacio PCA (TF-IDF)")
plt.show()

### Interpretación de la visualización:

- Los puntos azules son documentos de **deportes** (clase 0).
- Los puntos rojos son documentos de **política** (clase 1).
- La línea de separación es la frontera de decisión del SVM.
- Aunque hemos reducido a 2D con PCA, el SVM puede separar las clases razonablemente bien.
- Los números indican el índice del documento en el corpus original.

### 4.1. Experimentación con diferentes kernels de SVM

In [ ]:
kernels = ['linear', 'rbf', 'poly']

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, kernel in enumerate(kernels):
    if kernel == 'linear':
        svm_kernel = SVC(kernel=kernel, C=1.0, random_state=42)
    elif kernel == 'rbf':
        svm_kernel = SVC(kernel=kernel, C=1.0, gamma='scale', random_state=42)
    else:
        svm_kernel = SVC(kernel=kernel, degree=2, C=1.0, random_state=42)
    
    svm_kernel.fit(X_pca, y)
    
    plt.sca(axes[i])
    plot_decision_boundary(X_pca, y, svm_kernel, f"SVM kernel={kernel}")

plt.tight_layout()
plt.show()

---
## 5. Evaluación Completa de Modelos

Evaluamos todos los modelos con los datos de entrenamiento (para este ejemplo pequeño) para comparar.

In [ ]:
# Predecimos sobre el mismo corpus
y_pred_manual = nb_manual.predict(corpus)

# Para MultinomialNB, usamos el vectorizador ya entrenado
X_vec_all = vectorizer.transform(corpus)
y_pred_mnb = mnb.predict(X_vec_all)

# Para GaussianNB
X_tfidf_all = tfidf_vectorizer.transform(corpus).toarray()
y_pred_gnb = gnb.predict(X_tfidf_all)

# Para SVM (en espacio original, no PCA)
svm_original = LinearSVC(random_state=42)
svm_original.fit(X_vec_all, y)
y_pred_svm = svm_original.predict(X_vec_all)

# Calcular accuracy
print("=== Accuracy sobre el corpus de entrenamiento ===")
print(f"Naive Bayes Manual: {accuracy_score(y, y_pred_manual):.2%}")
print(f"MultinomialNB: {accuracy_score(y, y_pred_mnb):.2%}")
print(f"GaussianNB: {accuracy_score(y, y_pred_gnb):.2%}")
print(f"SVM Lineal: {accuracy_score(y, y_pred_svm):.2%}")

---
## 6. Ejercicio para el Estudiante

1. Añade un nuevo documento al corpus: "el partido de fútbol y la política no se mezclan". ¿Cómo crees que lo clasificarán los modelos?
2. Prueba a cambiar el parámetro alpha en Naive Bayes (suavizado Laplace) y observa cómo afecta a las predicciones.
3. Experimenta con diferentes valores de C en SVM y observa cómo cambia la frontera.
4. ¿Por qué GaussianNB no funciona bien con texto? (Pista: ¿qué tipo de distribución asume?)

In [ ]:
# Espacio para el estudiante
# nuevo_doc = "el partido de fútbol y la política no se mezclan"
# ...

---
## 7. Conclusiones

En este notebook hemos explorado los fundamentos de los modelos clásicos para clasificación de texto:

✔️ **Naive Bayes**:
- Implementación manual basada en frecuencias y teorema de Bayes.
- Comparación con MultinomialNB (ideal para texto) y GaussianNB (no recomendado).
- El suavizado Laplace evita probabilidades cero.

✔️ **SVM**:
- Visualización de la frontera de decisión en espacio reducido con PCA.
- Comparación de kernels lineal, RBF y polinomial.
- El kernel lineal suele ser suficiente para texto.

✔️ **Comparación**:
- Naive Bayes es rápido y simple, pero asume independencia.
- SVM es más robusto y suele dar mejor rendimiento.
- La elección depende del tamaño de datos y necesidad de interpretabilidad.

**Lección clave**: Los modelos clásicos siguen siendo herramientas valiosas, especialmente como líneas base y en entornos con recursos limitados.

En el próximo notebook (NB2) aplicaremos estos conceptos a un problema real de análisis de sentimiento.

---
**Fin del Notebook Conceptual - Semana 5 (NLP)**